<a href="https://colab.research.google.com/github/hemanth2406/moshi/blob/main/moshi_telugu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
# Install the libraries if needed
!pip install -q datasets soundfile huggingface_hub ai4bharat-transliteration

from datasets import load_dataset
from huggingface_hub import notebook_login

# Recommended: Log in to Hugging Face to avoid rate limits and warnings
# Running this will prompt you to enter your Hugging Face Access Token
notebook_login()

### 1. Load the Telugu Train Split
We explicitly select the `'train'` split of the `'telugu'` configuration of the `ai4bharat/IndicVoices` dataset.

In [ ]:
from huggingface_hub import login

login()

from datasets import load_dataset

print("Initializing streaming dataset...")
print("Starting dataset load...")

from datasets import load_dataset
from collections import Counter
ds = load_dataset("ai4bharat/IndicVoices", "telugu", split="train", streaming=True)
counts = Counter()
conv_rows = []
for i, row in enumerate(ds):
    sc = str(row.get("scenario") or "")
    counts[sc] += 1
    if sc.lower() == "conversation" and len(conv_rows) < 100:
        conv_rows.append(row)
    if len(conv_rows) >= 100 and i > 3000:
        break
print("scenario counts (sample):", counts)
print("conversation clips kept:", len(conv_rows))
print("example:", conv_rows[0]["scenario"], conv_rows[0]["duration"])

### 2. Build stereo on Drive (left = agent, right = user)

Pair clip 0 with 1, 2 with 3, … so each file has two different speakers.

**Roman Telugu:** Native Telugu script from IndicVoices (`text` / `normalized`) is converted to Latin script with `ai4bharat-transliteration` (`XlitEngine`, `src_script_type="indic"`, `lang_code="te"`). Moshi's text stream then gets Roman tokens instead of fragmented Telugu Unicode bytes.

**Turn-taking:** User (right) speaks first with silence on left; then agent (left) replies with silence on right. Agent-side alignments use romanized words with evenly spaced timestamps over the agent turn (replace later with forced ASR alignment when available).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, re
import numpy as np
import soundfile as sf
import torch
import torchaudio
from ai4bharat.transliteration import XlitEngine

out = "/content/drive/MyDrive/moshi_telugu/stereo_mvp_roman"
os.makedirs(out, exist_ok=True)

# Indic -> Roman (Telugu). Load once; reuse for all clips.
xlit = XlitEngine(src_script_type="indic", beam_width=4, rescore=False)


def to_24k_mono(row):
    audio = row.get("audio_filepath") or row["audio"]
    wav = torch.as_tensor(audio["array"], dtype=torch.float32)
    if wav.ndim == 2:
        wav = wav.mean(0)
    sr = audio["sampling_rate"]
    if sr != 24000:
        wav = torchaudio.functional.resample(wav.unsqueeze(0), sr, 24000).squeeze(0)
    return wav.numpy()


def row_text(row):
    return (row.get("normalized") or row.get("text") or row.get("verbatim") or "").strip()


def romanize_word(word):
    clean = re.sub(r"[^\u0C00-\u0C7Fa-zA-Z0-9']+", "", word)
    if not clean:
        return None
    # Already Latin (English loanwords / numbers)
    if re.fullmatch(r"[A-Za-z0-9']+", clean):
        return clean.lower()
    try:
        out = xlit.translit_word(clean, lang_code="te", topk=1)
        # Indic->Roman returns a list; Roman->Indic returns a dict
        if isinstance(out, dict):
            cand = out.get("te") or next(iter(out.values()))
            roman = cand[0] if isinstance(cand, list) else cand
        else:
            roman = out[0] if isinstance(out, list) else out
        return str(roman).lower().strip() or None
    except Exception as e:
        print("xlit skip:", clean, e)
        return None


def romanize_sentence(text):
    words = []
    for tok in text.split():
        r = romanize_word(tok)
        if r:
            words.append(r)
    return words


def agent_alignments(roman_words, t0, t1):
    """Evenly space roman words over the agent turn [t0, t1]."""
    if not roman_words or t1 <= t0:
        return [["<utt>", [float(t0), float(t1)], "SPEAKER_MAIN"]]
    n = len(roman_words)
    span = (t1 - t0) / n
    return [
        [w, [round(t0 + i * span, 3), round(t0 + (i + 1) * span, 3)], "SPEAKER_MAIN"]
        for i, w in enumerate(roman_words)
    ]


def build_turn_taking_stereo(user_wav, agent_wav, gap_sec=0.3):
    """Right=user first, then Left=agent. Opposite channel is silence."""
    gap = np.zeros(int(gap_sec * 24000), dtype=np.float32)
    left = np.concatenate([np.zeros_like(user_wav), gap, agent_wav])
    right = np.concatenate([user_wav, gap, np.zeros_like(agent_wav)])
    t_user = len(user_wav) / 24000
    t_agent0 = t_user + gap_sec
    t_agent1 = t_agent0 + len(agent_wav) / 24000
    stereo = np.stack([left, right], axis=1)
    return stereo, t_agent0, t_agent1, len(left) / 24000


def write_pair(n, user_row, agent_row, prefix):
    user_wav = to_24k_mono(user_row)
    agent_wav = to_24k_mono(agent_row)
    stereo, t0, t1, dur = build_turn_taking_stereo(user_wav, agent_wav)
    roman_words = romanize_sentence(row_text(agent_row))
    path = f"{out}/{prefix}_{n:03d}.wav"
    sf.write(path, stereo, 24000)
    with open(path.replace(".wav", ".json"), "w", encoding="utf-8") as f:
        json.dump({"alignments": agent_alignments(roman_words, t0, t1)}, f, ensure_ascii=False)
    return {"path": path, "duration": dur, "roman_preview": " ".join(roman_words[:8])}


jsonl_train, jsonl_eval = [], []
for n, i in enumerate(range(0, 80, 2)):  # 40 train: odd=user, even=agent
    meta = write_pair(n, conv_rows[i], conv_rows[i + 1], "train")
    jsonl_train.append({"path": meta["path"], "duration": meta["duration"]})
    if n < 3:
        print("train", n, "roman:", meta["roman_preview"])

for n, i in enumerate(range(80, 100, 2)):  # 10 eval hold-out
    meta = write_pair(n, conv_rows[i], conv_rows[i + 1], "eval")
    jsonl_eval.append({"path": meta["path"], "duration": meta["duration"]})


def write_jsonl(name, rows):
    with open(f"{out}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


write_jsonl("train.jsonl", jsonl_train)
write_jsonl("eval.jsonl", jsonl_eval)
print("wrote", out, "train", len(jsonl_train), "eval", len(jsonl_eval))
print("sample alignment:", open(jsonl_train[0]["path"].replace(".wav", ".json"), encoding="utf-8").read()[:400])

### 3. Extract Mimi Codec Tokens
Below is the code to load the pretrained Mimi model, resample the audio to the required sample rate (Mimi typically uses 24kHz), and extract the audio tokens.

In [ ]:
import sys
import os

# Fallback installation to resolve the CUDA 13 vs CUDA 12 mismatch on Google Colab
try:
    import torch
    import torchaudio
except OSError as e:
    print("CUDA/binary mismatch detected. Re-installing CUDA-12 compatible torch/torchaudio and moshi...")
    !pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
    !pip install --force-reinstall moshi
    import site
    from importlib import reload
    reload(site)
    import torch
    import torchaudio

# Ensure moshi is installed and loaded
try:
    from moshi.models import getters
except ImportError:
    print("Moshi module not found. Installing...")
    !pip install -q moshi
    import site
    from importlib import reload
    reload(site)
    from moshi.models import getters

# Load the pretrained Mimi codec model
device = "cuda" if torch.cuda.is_available() else "cpu"
mimi_model = getters.get_mimi()
mimi_model.to(device)
mimi_model.eval()

def encode_audio_to_mimi(audio_array, sampling_rate):
    # Convert numpy array to torch tensor
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0) # [1, samples]

    # Mimi expects 24kHz audio. Resample if necessary
    target_sample_rate = 24000
    if sampling_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    waveform = waveform.unsqueeze(0).to(device) # Shape: [1, 1, samples] (batch, channels, samples)

    with torch.no_grad():
        # Encode the waveform to discrete codes
        codes = mimi_model.encode(waveform)
        # codes shape: [batch, codebooks, timesteps]
    return codes.squeeze(0).cpu() # [codebooks, timesteps]

# Use the filtered sample
audio_data = sample['audio_filepath']['array'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['array']
sr = sample['audio_filepath']['sampling_rate'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['sampling_rate']

tokens = encode_audio_to_mimi(audio_data, sr)
print(f"Extracted Mimi tokens shape: {tokens.shape} (Codebooks: {tokens.shape[0]}, Timesteps: {tokens.shape[1]})")